In [ ]:
"""
Atlantis Occupation Prediction + Feature Importance Export
----------------------------------------------------------
This script helps the High Council of Atlantis restore the corrupted census records.
We’ll train a Random Forest model, tune it, validate it, predict hidden occupations,
and export both predictions and feature importance rankings for review.
"""

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

# -----------------------------
# Step 1: Load the data
# -----------------------------
print("🌊 Welcome to Atlantis! Loading census data...")
train = pd.read_csv("atlantis_citizens_final.csv")
test = pd.read_csv("test_atlantis_hidden.csv")

# -----------------------------
# Step 2: Prepare the features
# -----------------------------
print("🧹 Cleaning data: removing IDs and bio hashes...")
X = train.drop(columns=["Citizen_ID", "Bio_Hash", "Occupation"])
y = train["Occupation"]

X_test = test.drop(columns=["Citizen_ID", "Bio_Hash"])

print("🔄 Encoding categorical features into numbers...")
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)

# Align train and test columns so they match
X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

# -----------------------------
# Step 3: Hyperparameter tuning
# -----------------------------
print("⚙️ Tuning the Random Forest model... patience, Council.")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": ["balanced"]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("✅ Best parameters found:", grid_search.best_params_)

# -----------------------------
# Step 4: Train the best model
# -----------------------------
print("🌱 Training the best Random Forest model...")
best_rf = grid_search.best_estimator_
best_rf.fit(X_train, y_train)

# -----------------------------
# Step 5: Validate performance
# -----------------------------
print("📊 Validating model performance...")
y_pred = best_rf.predict(X_val)
print("Here’s how well the model performs:\n")
print(classification_report(y_val, y_pred))

# -----------------------------
# Step 6: Predict hidden occupations
# -----------------------------
print("🔮 Predicting occupations for hidden citizens...")
test["Occupation"] = best_rf.predict(X_test)
test[["Citizen_ID", "Occupation"]].to_csv("submission.csv", index=False)
print("📂 Predictions saved to submission.csv")

# -----------------------------
# Step 7: Feature importance visualization + export
# -----------------------------
print("🌟 Highlighting the top lifestyle features that influence occupation...")

importances = best_rf.feature_importances_
feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

# Save full ranking to CSV
feature_importance_df.to_csv("feature_importance.csv", index=False)
print("📂 Feature importance rankings saved to feature_importance.csv")

# Plot top 20 features
top20 = feature_importance_df.head(20)
plt.figure(figsize=(10,6))
plt.bar(range(len(top20)), top20["Importance"], align="center", color="teal")
plt.xticks(range(len(top20)), top20["Feature"], rotation=90)
plt.title("Top 20 Features Influencing Occupation in Atlantis")
plt.tight_layout()
plt.show()

print("✨ Done! The High Council now has both predictions and feature importance rankings.")